# Food Image-Based Weight and Macronutrient Estimation (Kaggle orchestrator)

Thin orchestrator that imports the reusable `src/foodmacros` package. Attach:

- `hari31416/food-101` (images)
- `sanadalali/food-101-nutritional-information` (macros)

Optional: a Nutrition5k mirror (mass regressor) and a food-detection YOLO `.pt`.
Every stage degrades honestly: missing pieces are reported, never faked.

In [ ]:
import os, sys

print('Running on Kaggle' if os.path.isdir('/kaggle/input') else 'Not running on Kaggle (local run)')

# Make the packaged source importable (Kaggle path first, then repo-relative).
for candidate in ('/kaggle/working/Final_Year_Project/src', 'src', '../src'):
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
print('src on path:', [p for p in sys.path if p.endswith('src')][:1])

In [ ]:
from pathlib import Path
from foodmacros.config import load_config
from foodmacros.env_checks import get_env_report, format_report, configure_mixed_precision, default_working_dir

config = load_config()
print(format_report(get_env_report()))
configure_mixed_precision(config.runtime.mixed_precision)
run_dir = Path(config.runtime.output_dir) if config.runtime.output_dir else default_working_dir()
print('run_dir:', run_dir, '| mode:', config.runtime.mode)

In [ ]:
from foodmacros.data.discovery import resolve_inputs

resolved = resolve_inputs(config)
print(resolved.summary())

## Train the classifier (optional)

Skip if you already have `classifier.keras`. Set `config.runtime.mode = 'full'`
for a full run; `'quick'` uses one short epoch.

In [ ]:
from foodmacros.data.splits import make_splits, build_class_names, save_class_names
from foodmacros.classification.train import train_classifier

if resolved.image_root:
    class_names = build_class_names(
        resolved.image_root, resolved.validation_root, config.data.class_names_file
    )
    save_class_names(config.data.class_names_file or (run_dir / 'class_names.json'), class_names)
    splits = make_splits(
        resolved.image_root,
        class_names,
        validation_root=resolved.validation_root,
        val_fraction=config.splits.val_fraction,
        seed=config.splits.shuffle_seed,
        test_source=config.splits.test_source,
    )
    print(splits.summary())
    result = train_classifier(
        config, class_names, splits.train, splits.val, output_dir=run_dir
    )
    print(result.summary())
else:
    print('No image root; skipping training.')

## Build the pipeline and analyze a sample image

In [ ]:
from foodmacros.pipeline import FoodAnalysisPipeline
from foodmacros.reporting import format_analysis, analysis_to_dict

pipeline, resolved = FoodAnalysisPipeline.from_discovery(config, model_dir=run_dir)

sample = None
if resolved.image_root:
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        found = sorted(Path(resolved.image_root).rglob(ext))
        if found:
            sample = str(found[0])
            break

if sample:
    result = pipeline.analyze(sample, measured_grams=None, top_k=config.ui.max_top_k)
    print(format_analysis(result))
    analysis_to_dict(result)
else:
    print('No sample image found; pipeline constructed but not exercised.')

## Launch the Gradio app (optional)

Enable a public link by setting `config.ui.share = True`. Requires `pip install gradio`.

In [ ]:
from foodmacros.env_checks import has_package

if has_package('gradio'):
    project_root = str(run_dir.parent) if run_dir.name == 'outputs' else str(run_dir)
    if os.path.isdir('/kaggle/working/Final_Year_Project'):
        project_root = '/kaggle/working/Final_Year_Project'
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    from app.gradio_app import build_interface
    demo = build_interface(pipeline)
    demo.launch(share=config.ui.share)
else:
    print('gradio not installed; skipping UI launch.')